In [7]:
# ============================================================
# CELL 1 — IMPORTS AND PROJECT PATHS
# ============================================================

import os
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import py4dgeo


# ------------------------------------------------------------
# Repository and data paths
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "jupyter":
    REPO_DIR = CURRENT_DIR.parent
else:
    REPO_DIR = CURRENT_DIR


DATA_DIR = REPO_DIR / "kijkduin"
POINTCLOUD_DIR = DATA_DIR / "pointclouds"

RESULTS_DIR = REPO_DIR / "results"
TABLES_DIR = REPO_DIR / "tables"
FIGURES_DIR = REPO_DIR / "figures"


for folder in [
    RESULTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
]:
    folder.mkdir(
        exist_ok=True
    )


# ------------------------------------------------------------
# Validate input data location
# ------------------------------------------------------------

if not POINTCLOUD_DIR.is_dir():
    raise FileNotFoundError(
        f"Point-cloud directory not found:\n"
        f"{POINTCLOUD_DIR}"
    )


print(
    "Point-cloud directory:"
)

print(
    POINTCLOUD_DIR
)

Point-cloud directory:
C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\pointclouds


In [8]:
# ============================================================
# CELL 2 — DISCOVER EPOCH FILES AND TIMESTAMPS
# ============================================================

pc_list = sorted([
    file_name
    for file_name in os.listdir(
        POINTCLOUD_DIR
    )
    if file_name.lower().endswith(
        ".laz"
    )
])


if len(pc_list) == 0:
    raise FileNotFoundError(
        "No LAZ files found."
    )


# ------------------------------------------------------------
# Extract timestamps from filenames
# ------------------------------------------------------------

timestamps = []


for file_name in pc_list:

    stem = Path(
        file_name
    ).stem

    parts = stem.split("_")


    if len(parts) < 3:
        raise ValueError(
            f"Unexpected epoch filename format: "
            f"{file_name}"
        )


    timestamp_string = "_".join(
        parts[1:]
    )


    try:

        timestamp = datetime.strptime(
            timestamp_string,
            "%y%m%d_%H%M%S"
        )

    except ValueError as exc:

        raise ValueError(
            f"Could not extract timestamp from "
            f"filename: {file_name}"
        ) from exc


    timestamps.append(
        timestamp
    )


# ------------------------------------------------------------
# Reference and comparison epochs
# ------------------------------------------------------------

reference_epoch_file = (
    POINTCLOUD_DIR
    / pc_list[0]
)


reference_timestamp = (
    timestamps[0]
)


comparison_epoch_files = [
    POINTCLOUD_DIR
    / file_name
    for file_name in pc_list[1:]
]


comparison_timestamps = (
    timestamps[1:]
)


# ------------------------------------------------------------
# Dataset metadata
# ------------------------------------------------------------

metadata = pd.DataFrame({
    "epoch_index":
        np.arange(
            len(pc_list)
        ),

    "filename":
        pc_list,

    "timestamp":
        timestamps,
})


display(
    metadata.head()
)


print(
    "Number of epochs:",
    len(pc_list)
)

,epoch_index,filename,timestamp
0,0,kijkduin_170117_120041.laz,2017-01-17 12:00:41
1,1,kijkduin_170118_000050.laz,2017-01-18 00:00:50
2,2,kijkduin_170120_120036.laz,2017-01-20 12:00:36
3,3,kijkduin_170121_000046.laz,2017-01-21 00:00:46
4,4,kijkduin_170121_120055.laz,2017-01-21 12:00:55


Number of epochs: 160


In [9]:
# ============================================================
# CELL 3 — CREATE SPATIOTEMPORAL ANALYSIS
# ============================================================

analysis_file = (
    DATA_DIR
    / "kijkduin.zip"
)


if analysis_file.exists():
    analysis_file.unlink()


analysis = (
    py4dgeo.SpatiotemporalAnalysis(
        str(
            analysis_file
        ),
        force=True
    )
)


reference_epoch = (
    py4dgeo.read_from_las(
        str(
            reference_epoch_file
        )
    )
)


reference_epoch.timestamp = (
    reference_timestamp
)


analysis.reference_epoch = (
    reference_epoch
)


print(
    "Reference epoch:",
    reference_epoch_file.name
)

print(
    "Reference points:",
    reference_epoch.cloud.shape[0]
)

[2026-08-23 12:51:44][INFO] Creating analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 12:51:44][INFO] Reading point cloud from file 'C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\pointclouds\kijkduin_170117_120041.laz'
[2026-08-23 12:51:45][INFO] Building KDTree structure with leaf parameter 10
[2026-08-23 12:51:45][INFO] Saving epoch to file 'C:\Users\tenbi\AppData\Local\Temp\tmpysnwryx6\reference_epoch.zip'
[2026-08-23 12:51:45][INFO] Saving a file without normals.
Reference epoch: kijkduin_170117_120041.laz
Reference points: 215550


In [10]:
# ============================================================
# CELL 4 — COREPOINTS AND M3C2 CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# M3C2 parameters
# ------------------------------------------------------------

NORMAL_RADII = (5.0,)
CYL_RADIUS = 1.0
MAX_DISTANCE = 10.0
REGISTRATION_ERROR = 0.019


# ------------------------------------------------------------
# Corepoints
# ------------------------------------------------------------

analysis.corepoints = (
    reference_epoch.cloud
)


# ------------------------------------------------------------
# M3C2 configuration
# ------------------------------------------------------------

analysis.m3c2 = py4dgeo.M3C2(
    normal_radii=NORMAL_RADII,
    cyl_radius=CYL_RADIUS,
    max_distance=MAX_DISTANCE,
    registration_error=REGISTRATION_ERROR,
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print(
    "Corepoints:",
    analysis.corepoints.cloud.shape[0]
)


print(
    "\nM3C2 configuration:"
)

print(
    "normal_radii =",
    NORMAL_RADII
)

print(
    "cyl_radius =",
    CYL_RADIUS
)

print(
    "max_distance =",
    MAX_DISTANCE
)

print(
    "registration_error =",
    REGISTRATION_ERROR
)

[2026-08-23 12:53:05][INFO] Initializing Epoch object from given point cloud
[2026-08-23 12:53:05][INFO] Building KDTree structure with leaf parameter 10
[2026-08-23 12:53:06][INFO] Saving epoch to file 'C:\Users\tenbi\AppData\Local\Temp\tmpd241vqww\corepoints.zip'
[2026-08-23 12:53:06][INFO] Saving a file without normals.
Corepoints: 215550

M3C2 configuration:
normal_radii = (5.0,)
cyl_radius = 1.0
max_distance = 10.0
registration_error = 0.019


In [11]:
# ============================================================
# CELL 5 — ADD COMPARISON EPOCHS
# ============================================================

# ------------------------------------------------------------
# Validate epoch/timestamp correspondence
# ------------------------------------------------------------

if (
    len(comparison_epoch_files)
    !=
    len(comparison_timestamps)
):
    raise ValueError(
        "The number of comparison epoch files does not "
        "match the number of comparison timestamps."
    )


# ------------------------------------------------------------
# Add comparison epochs
# ------------------------------------------------------------

start_time = time.time()


for index, (
    epoch_file,
    timestamp
) in enumerate(
    zip(
        comparison_epoch_files,
        comparison_timestamps
    ),
    start=1
):

    epoch = py4dgeo.read_from_las(
        str(
            epoch_file
        )
    )

    epoch.timestamp = (
        timestamp
    )


    analysis.add_epochs(
        epoch
    )


    print(
        f"{index:03d}/"
        f"{len(comparison_epoch_files)} "
        f"{epoch_file.name}"
    )


# ------------------------------------------------------------
# Processing summary
# ------------------------------------------------------------

elapsed = (
    time.time()
    -
    start_time
)


print(
    "\nM3C2 time series complete."
)


print(
    "Distances shape:",
    analysis.distances.shape
)


print(
    "Uncertainties shape:",
    analysis.uncertainties.shape
)


print(
    f"Elapsed time: {elapsed:.2f} s"
)

[2026-08-23 12:57:40][INFO] Reading point cloud from file 'C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\pointclouds\kijkduin_170118_000050.laz'
[2026-08-23 12:57:40][INFO] Removing intermediate results from the analysis file C:\Users\tenbi\py4dgeo-kalmanfiltering\kijkduin\kijkduin.zip
[2026-08-23 12:57:40][INFO] Starting: Adding epoch 1/1 to analysis object
[2026-08-23 12:57:40][INFO] Building KDTree structure with leaf parameter 10
[2026-08-23 12:57:41][INFO] Finished in 1.4886s: Adding epoch 1/1 to analysis object
[2026-08-23 12:57:41][INFO] Starting: Rearranging space-time array in memory
[2026-08-23 12:57:42][INFO] Finished in 0.7424s: Rearranging space-time array in memory
[2026-08-23 12:57:42][INFO] Starting: Updating disk-based analysis archive with new epochs
[2026-08-23 12:57:42][INFO] Finished in 0.1584s: Updating disk-based analysis archive with new epochs
001/159 kijkduin_170118_000050.laz
[2026-08-23 12:57:42][INFO] Reading point cloud from file 'C:\Users\tenbi\py4dgeo-

In [12]:
# ============================================================
# CELL 6 — TEMPORAL SMOOTHING AND FINAL VERIFICATION
# ============================================================

# ------------------------------------------------------------
# Smoothing parameter
# ------------------------------------------------------------

SMOOTHING_WINDOW = 14


# ------------------------------------------------------------
# Analysis timestamps
# ------------------------------------------------------------

timestamps_analysis = [
    analysis.reference_epoch.timestamp
    + timedelta
    for timedelta
    in analysis.timedeltas
]


# ------------------------------------------------------------
# Temporal smoothing
# ------------------------------------------------------------

analysis.smoothed_distances = (
    py4dgeo.temporal_averaging(
        analysis.distances,
        smoothing_window=(
            SMOOTHING_WINDOW
        )
    )
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

print(
    "Raw M3C2 shape:",
    analysis.distances.shape
)


print(
    "Smoothed M3C2 shape:",
    analysis.smoothed_distances.shape
)


print(
    "Number of timestamps:",
    len(
        timestamps_analysis
    )
)


print(
    "\nPreparation complete."
)

[2026-08-23 15:06:43][INFO] Restoring epoch from file 'C:\Users\tenbi\AppData\Local\Temp\tmpenvu128v\reference_epoch.zip'
[2026-08-23 15:06:44][INFO] Starting: Smoothing temporal data
[2026-08-23 15:07:25][INFO] Finished in 40.6085s: Smoothing temporal data
Raw M3C2 shape: (215550, 159)
Smoothed M3C2 shape: (215550, 159)
Number of timestamps: 159

Preparation complete.
